In [2]:
!pip install transformers datasets -q

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Load data
dataset = load_dataset("imdb")
train = dataset["train"].select(range(100))

# Model
tokenizer = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")
model = AutoModelForSequenceClassification.from_pretrained("prajjwal1/bert-tiny", num_labels=2)

# FIXED tokenize
def tokenize(x):
    return tokenizer(
        x["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train = train.map(tokenize)

# IMPORTANT
train = train.rename_column("label", "labels")
train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Train
args = TrainingArguments(
    output_dir="out",
    per_device_train_batch_size=8,
    num_train_epochs=1
)

trainer = Trainer(model=model, args=args, train_dataset=train)

trainer.train()

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect i

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=13, training_loss=0.8615452693058894, metrics={'train_runtime': 2.0664, 'train_samples_per_second': 48.394, 'train_steps_per_second': 6.291, 'total_flos': 31762176000.0, 'train_loss': 0.8615452693058894, 'epoch': 1.0})

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer, TrainingArguments

# Load small dataset
dataset = load_dataset("squad")
data = dataset["train"].select(range(100))

# Model
model = AutoModelForQuestionAnswering.from_pretrained("prajjwal1/bert-tiny")
tokenizer = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")

# FIXED preprocess
def preprocess(x):
    tokens = tokenizer(
        x["question"],
        x["context"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    #ADD DUMMY ANSWER POSITIONS
    tokens["start_positions"] = [0] * len(tokens["input_ids"])
    tokens["end_positions"] = [1] * len(tokens["input_ids"])

    return tokens

data = data.map(preprocess, batched=True)

# IMPORTANT
data.set_format("torch", columns=["input_ids", "attention_mask", "start_positions", "end_positions"])

# Train
args = TrainingArguments(
    output_dir="qa",
    per_device_train_batch_size=8,
    num_train_epochs=1
)

trainer = Trainer(model=model, args=args, train_dataset=data)

trainer.train()

print("QA training runs successfully")

Loading weights:   0%|          | 0/37 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

QA training runs successfully
